In [1]:
#Resume Screening System

In [2]:
import pandas as pd
import numpy as np
import re

import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
dataset=pd.read_csv('Project 1 file/Resume/Resume.csv')

In [4]:
dataset.head(3)

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [5]:
dataset.isnull().sum()

ID             0
Resume_str     0
Resume_html    0
Category       0
dtype: int64

In [6]:
dataset = dataset.drop("Resume_html", axis=1)

In [7]:
dataset["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
FINANCE                   118
ENGINEERING               118
ACCOUNTANT                118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64

In [8]:
print(dataset.iloc[0])

ID                                                     16852973
Resume_str             HR ADMINISTRATOR/MARKETING ASSOCIATE\...
Category                                                     HR
Name: 0, dtype: object


In [9]:
stopwords=set(stopwords.words('english'))

In [10]:
lemmatizer=WordNetLemmatizer()

In [11]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

In [12]:
#Preprocess function 
def text_preprocess(text):
    text=re.sub('[^A-Za-z]',' ',text)
    text=text.lower()
    words=text.split()
    words = [
        word for word in words
        if word not in stop_words
    ]
    words=[lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

In [13]:
#Seeing preprocess for the 1st resume
original_text=dataset['Resume_str'].iloc[0]
preprocessed_text=text_preprocess(original_text)

In [14]:
print(original_text[:50]),print(preprocessed_text[:50])

         HR ADMINISTRATOR/MARKETING ASSOCIATE

HR 
hr administrator marketing associate hr administra


(None, None)

In [15]:
dataset['clean_resume']=dataset['Resume_str'].apply(text_preprocess)

In [16]:
job_description = """
We are looking for a Machine Learning Engineer with experience
in Python, Machine Learning, Pandas, NumPy, Scikit-learn,
Natural Language Processing, SQL and Deep Learning.
The candidate should have experience in developing machine
learning models, data preprocessing, model evaluation and
working with large datasets.
"""

In [17]:
clean_job_description=text_preprocess(job_description)

In [18]:
print(clean_job_description)

looking machine learning engineer experience python machine learning panda numpy scikit learn natural language processing sql deep learning candidate experience developing machine learning model data preprocessing model evaluation working large datasets


In [19]:
vectorizer=TfidfVectorizer()

In [20]:
all_text=[job_description]+dataset['clean_resume'].tolist()

In [21]:
tf=TfidfVectorizer()
x=tf.fit_transform(all_text)

In [22]:
job_vector=x[0]

In [23]:
resumes=x[1:]

In [24]:
similarity_scores = cosine_similarity(
    job_vector,
    resumes
)

In [25]:
dataset["match_score_"]=similarity_scores.flatten()

In [26]:
dataset.head()

,ID,Resume_str,Category,clean_resume,match_score_
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR,hr administrator marketing associate hr admini...,0.004277
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...",HR,hr specialist u hr operation summary versatile...,0.008418
2,33176873,HR DIRECTOR Summary Over 2...,HR,hr director summary year experience recruiting...,0.006056
3,27018550,HR SPECIALIST Summary Dedica...,HR,hr specialist summary dedicated driven dynamic...,0.001626
4,17812897,HR MANAGER Skill Highlights ...,HR,hr manager skill highlight hr skill hr departm...,0.000692


In [27]:
dataset["percentage_match_score"]=dataset["match_score_"]*100

In [28]:
dataset["percentage_match_score"].value_counts()

percentage_match_score
0.000000    7
0.300518    2
1.454430    2
0.427714    1
0.841822    1
           ..
0.058418    1
1.297553    1
0.200345    1
0.980076    1
0.228078    1
Name: count, Length: 2476, dtype: int64

In [29]:
ranked_resumes = dataset.sort_values(
    by="percentage_match_score",
    ascending=False
)

In [30]:
ranked_resumes.head(5)

,ID,Resume_str,Category,clean_resume,match_score_,percentage_match_score
2291,12777487,ONLINE LEARNING COORDINATOR - PROGRAM...,ARTS,online learning coordinator program training s...,0.112856,11.285575
746,26958533,"ENGINEER Summary Work Ethic,...",HEALTHCARE,engineer summary work ethic personal ownership...,0.109730,10.973022
2153,34953092,"CORPORATE BANKING ASSISTANT, INTERN ...",BANKING,corporate banking assistant intern objective d...,0.103573,10.357301
194,18835363,PROGRAM MANAGER & DESIGNER ...,DESIGNER,program manager designer professional summary ...,0.097909,9.790940
1762,12011623,ENGINEERING AND QUALITY TECHNICIAN ...,ENGINEERING,engineering quality technician career overview...,0.094979,9.497861


In [31]:
best_candidate = ranked_resumes.iloc[0]

In [32]:
best_candidate

ID                                                                 12777487
Resume_str                         ONLINE LEARNING COORDINATOR - PROGRAM...
Category                                                               ARTS
clean_resume              online learning coordinator program training s...
match_score_                                                       0.112856
percentage_match_score                                            11.285575
Name: 2291, dtype: object